# Séance 6 — Modélisation supervisée et calibration
*Session 6 — Supervised modelling and calibration*

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/fraude-360-cours/blob/main/cours/laboratoires/06_modelisation_calibration.ipynb)

**Cours :** Science des données appliquée : détection de la fraude transactionnelle (45 h) · **Chargé de cours :** Noé Carème Fouotsa Manfouo · **Durée :** 3 h

| Parcours de formation | Cette séance |
|---|---|
| Gestion — UdeS École de gestion (SIMQG), Bishop's Williams School | ● séance complète |
| Informatique — UdeS maîtrise/bac en informatique (IA et science des données) | ● séance complète |
| Cégep / formation continue — Cégep de Sherbrooke (AEC), UdeS formation continue | ● séance complète |

**À la fin de cette séance, vous saurez :**

1. découper les données dans le **temps** (entraînement, validation, test) ;
2. entraîner une régression logistique et un gradient boosting dans des pipelines scikit-learn ;
3. évaluer avec la PR-AUC (aire sous la courbe précision-rappel) plutôt que l'exactitude ;
4. calibrer des probabilités (isotonique) et le vérifier avec le score de Brier.

**Mode d'emploi :** exécutez les cellules dans l'ordre (`Maj + Entrée`). Complétez les cellules `# VOTRE CODE ICI`, puis
lancez la cellule ✅ qui suit pour obtenir une rétroaction immédiate. Chaque exercice indique le ou les parcours concernés :
🟢 tous · 🟣 Gestion + Informatique · 🔷 Informatique · 🔶 Gestion. Les exercices 🔷/🔶 sont facultatifs pour les autres parcours.

In [ ]:
# --- Configuration (une fois par session) ----------------------------------------
import sys, pathlib, subprocess, shutil
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not pathlib.Path("fraude-360-cours").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/fraude-360-cours.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "fraude-360-cours/cours/requirements-colab.txt"], check=True)
    if shutil.which("java") is None:   # PySpark a besoin d'un Java 11, 17 ou 21
        subprocess.run("apt-get -qq install -y openjdk-17-jre-headless > /dev/null", shell=True, check=True)
    RACINE = pathlib.Path("fraude-360-cours").resolve()
else:
    ici = pathlib.Path.cwd().resolve()
    RACINE = ici.parents[1] if ici.name == "laboratoires" else ici
sys.path.insert(0, str(RACINE / "cours"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from outils_cours import preparer, config, table, verifier, verifier_proche, image, lire_json
pd.set_option("display.max_columns", 40)
print("Racine du projet :", RACINE)

In [ ]:
cfg = preparer("variables")
from fraude360.modeling import donnees, entrainement
jeux = donnees.decouper(cfg)
num, cat, cible = jeux.variables_numeriques, jeux.variables_categorielles, jeux.cible
for nom, d in [("entraînement", jeux.entrainement), ("validation", jeux.validation), ("test", jeux.test)]:
    print(f"{nom:<13} {len(d):>6} lignes  jours {d.jour_index.min():>2}–{d.jour_index.max():<2}  fraude {d[cible].mean():.3%}")
Xtr, ytr = jeux.entrainement[num + cat], jeux.entrainement[cible].values
Xva, yva = jeux.validation[num + cat], jeux.validation[cible].values
Xte, yte = jeux.test[num + cat], jeux.test[cible].values

**Pourquoi un découpage temporel ?** En production, le modèle prédit l'avenir à partir du passé. Un découpage aléatoire
mélange des transactions d'une même rafale de fraude dans l'entraînement et le test : la performance serait surestimée.

**Pourquoi la PR-AUC ?** Avec 0,45 % de fraude, la ROC-AUC reste élevée même pour un modèle médiocre. La PR-AUC part du taux
de base (≈ 0,0045) : c'est la référence d'un tirage au hasard.

In [ ]:
modeles = entrainement.construire_modeles(num, cat, cfg.seed)
print(list(modeles))
modeles["regression_logistique"]

### Exercice 6.1 — 🟢 Tous les parcours

Entraînez `modeles["regression_logistique"]` sur l'entraînement, puis calculez `pr_auc_lr`, la PR-AUC
(`average_precision_score`) sur la **validation**.

In [ ]:
# VOTRE CODE ICI
from sklearn.metrics import average_precision_score
lr = modeles["regression_logistique"]
pr_auc_lr = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier(pr_auc_lr > 10 * yva.mean(), "Le modèle doit faire beaucoup mieux que le hasard")

### Exercice 6.2 — 🟣 Gestion + Informatique

Entraînez `modeles["gradient_boosting"]` et calculez `pr_auc_gb` sur la validation. Lequel retiendriez-vous ?
Quel argument plaide pour garder la régression logistique comme **modèle de référence** ?

In [ ]:
# VOTRE CODE ICI
gb = modeles["gradient_boosting"]
pr_auc_gb = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier(pr_auc_gb > 10 * yva.mean())

**Calibration.** Les modèles entraînés avec `class_weight="balanced"` surestiment fortement les probabilités. Or le métier
a besoin de vraies probabilités : la **perte attendue** d'une transaction vaut `p × montant`. On recalibre sur la validation
(modèle de base figé), puis on mesure sur le test.

### Exercice 6.3 — 🟢 Tous les parcours

Calibrez `gb` avec `CalibratedClassifierCV(FrozenEstimator(gb), method="isotonic")` ajusté sur la **validation**.
Calculez `brier_avant` et `brier_apres` (`brier_score_loss`) sur le **test**.

In [ ]:
# VOTRE CODE ICI
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import brier_score_loss
brier_avant = ...
brier_apres = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier(brier_apres < brier_avant, "La calibration doit réduire le score de Brier")

**❓ Question (🔶 Gestion — décision d'affaires) —** Une transaction de 2 000 $ a une probabilité calibrée de fraude de 3 %; une autre de 40 $ a 30 %. Laquelle l'enquêteur devrait-il examiner en premier ? Pourquoi la calibration est-elle indispensable pour répondre ?

*Votre réponse ici.*

## Bilan

- Découpage temporel, PR-AUC et taux de base : les trois réflexes en classes déséquilibrées.
- Le gradient boosting gagne souvent, la régression logistique reste la référence interprétable.
- Calibrer avant de décider : les probabilités servent à calculer des pertes attendues.

**Pour aller plus loin :**

- `src/fraude360/modeling/entrainement.py` : comparaison des 3 modèles, calibration et enregistrement MLflow.

---
*Fraude-360 — données entièrement synthétiques. Cours conçu par Noé Carème Fouotsa Manfouo · licence MIT (code) et CC BY 4.0 (contenu pédagogique).*